**IMPORTANT:**
All file load and save paths are omitted in this python notebook and need to be filled in with the correct path from your own file locations.

In [15]:
!pip install captum

Defaulting to user installation because normal site-packages is not writeable


# Data Loading

Note: The dataset loaded in should already be split and processed correctly, with the outcome variable being "is_quartile_4" and variables such as the "subject_id", "eventname", "y_t", and "y_{t+1}" removed. Also, it is assumed that the dataset only contains the 8 summary CBCL features.

In [1]:
#START HERE
#Reload training, validation and testing data
import pandas as pd
ABCD_train = pd.read_parquet("Insert Here")
ABCD_val = pd.read_parquet("Insert Here")
ABCD_test = pd.read_parquet("Insert Here")

#Check all variable names to decide which ones to drop
print("Train Size: " + str(ABCD_train.shape))
print("Validation Size: " + str(ABCD_val.shape))
print("Test Size: " + str(ABCD_test.shape))
print(ABCD_train.columns.tolist())

Train Size: (22215, 158)
Validation Size: (2836, 158)
Test Size: (2803, 158)
['cbcl_scr_syn_anxdep_t', 'cbcl_scr_syn_withdep_t', 'cbcl_scr_syn_somatic_t', 'cbcl_scr_syn_social_t', 'cbcl_scr_syn_thought_t', 'cbcl_scr_syn_attention_t', 'cbcl_scr_syn_rulebreak_t', 'cbcl_scr_syn_aggressive_t', 'demo_comb_income_v2', 'parent_highest_education', 'demo_sex_v2_1', 'demo_sex_v2_2', 'demo_sex_v2_3', 'adi_percentile', 'interview_age', 'total_core', 'fam_enviro1_p', 'fam_enviro2r_p', 'fam_enviro3_p', 'fam_enviro4r_p', 'fam_enviro5_p', 'fam_enviro6_p', 'fam_enviro7r_p', 'fam_enviro8_p', 'fam_enviro9r_p', 'fes_youth_q1', 'fes_youth_q2', 'fes_youth_q3', 'fes_youth_q4', 'fes_youth_q5', 'fes_youth_q6', 'fes_youth_q7', 'fes_youth_q8', 'fes_youth_q9', 'neighborhood1r_p', 'neighborhood2r_p', 'neighborhood3r_p', 'neighborhood_crime_y', 'parent_monitor_q1_y', 'parent_monitor_q2_y', 'parent_monitor_q3_y', 'parent_monitor_q4_y', 'parent_monitor_q5_y', 'prosocial_q1_p', 'prosocial_q2_p', 'prosocial_q3_p', 'pro

In [2]:
#Define feature groups
cbcl_cols = [
    'cbcl_scr_syn_anxdep_t',
    'cbcl_scr_syn_withdep_t',
    'cbcl_scr_syn_somatic_t',
    'cbcl_scr_syn_social_t',
    'cbcl_scr_syn_thought_t',
    'cbcl_scr_syn_attention_t',
    'cbcl_scr_syn_rulebreak_t',
    'cbcl_scr_syn_aggressive_t'
]

y_train = ABCD_train["is_quartile_4"].values
y_val = ABCD_val["is_quartile_4"].values
y_test = ABCD_test["is_quartile_4"].values


ABCD_train = ABCD_train.drop(columns = ['is_quartile_4'])

mech_cols = [c for c in ABCD_train.columns if c not in cbcl_cols]

Only run the code segment below to train model/test it on the test dataset with CBCL features removed/set to 0

In [ ]:
ABCD_test = ABCD_test.loc[:, cbcl_cols] = 0

## Prepare Dataset

In [3]:
#Prepare dataset as tensors
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

#Separate data from label
X_mech_train = ABCD_train[mech_cols].values
X_cbcl_train = ABCD_train[cbcl_cols].values

X_mech_val = ABCD_val[mech_cols].values
X_cbcl_val = ABCD_val[cbcl_cols].values

X_mech_test = ABCD_test[mech_cols].values
X_cbcl_test = ABCD_test[cbcl_cols].values


#Create DataLoader Objects for each dataset
class ResidualDataset(Dataset):
    def __init__(self, X_mech, X_cbcl, y):
        self.X_mech = torch.tensor(X_mech, dtype=torch.float32)
        self.X_cbcl = torch.tensor(X_cbcl, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_mech[idx], self.X_cbcl[idx], self.y[idx]

g = torch.Generator()
g.manual_seed(SEED)

train_dataset = ResidualDataset(X_mech_train, X_cbcl_train, y_train)
val_dataset = ResidualDataset(X_mech_val, X_cbcl_val, y_val)
test_dataset = ResidualDataset(X_mech_test, X_cbcl_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Residual Model Training and Evaluation

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64), num_classes: int = 2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, num_classes)]  # logits for x classes
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # (batch, 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mech_model = MLP(input_dim=len(mech_cols), hidden_dims=(128, 64), num_classes=2).to(device)
cbcl_model = MLP(input_dim=len(cbcl_cols), hidden_dims=(8,), num_classes=2).to(device)

In [5]:
#Train Mechanistic Model
import copy

criterion = nn.CrossEntropyLoss()
optimizer_mech = torch.optim.Adam(mech_model.parameters(), lr=1e-3)

patience = 10
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

for epoch in range(100):
    mech_model.train()
    train_loss = 0.0
    n_train = 0

    for x_mech, x_cbcl, y in train_loader:
        x_mech = x_mech.to(device)
        y = y.to(device)

        optimizer_mech.zero_grad()
        logits = mech_model(x_mech)
        loss = criterion(logits, y)
        loss.backward()
        optimizer_mech.step()

        train_loss += loss.item() * y.size(0)
        n_train += y.size(0)

    train_loss /= n_train

    mech_model.eval()
    val_loss = 0.0
    n_val = 0

    with torch.no_grad():
        for x_mech, x_cbcl, y in val_loader:
            x_mech = x_mech.to(device)
            y = y.to(device)

            logits = mech_model(x_mech)
            loss = criterion(logits, y)

            val_loss += loss.item() * y.size(0)
            n_val += y.size(0)

    val_loss /= n_val

    print(f"[Mechanistic] epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f}")

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mech_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Mechanistic early stopping at epoch {epoch}")
            break

if best_state is not None:
    mech_model.load_state_dict(best_state)

[Mechanistic] epoch 000 | train_loss 0.3252 | val_loss 0.3121
[Mechanistic] epoch 001 | train_loss 0.3084 | val_loss 0.3147
[Mechanistic] epoch 002 | train_loss 0.3001 | val_loss 0.3216
[Mechanistic] epoch 003 | train_loss 0.2913 | val_loss 0.3308
[Mechanistic] epoch 004 | train_loss 0.2775 | val_loss 0.3464
[Mechanistic] epoch 005 | train_loss 0.2602 | val_loss 0.3675
[Mechanistic] epoch 006 | train_loss 0.2383 | val_loss 0.4223
[Mechanistic] epoch 007 | train_loss 0.2157 | val_loss 0.4635
[Mechanistic] epoch 008 | train_loss 0.1916 | val_loss 0.5270
[Mechanistic] epoch 009 | train_loss 0.1645 | val_loss 0.6058
[Mechanistic] epoch 010 | train_loss 0.1443 | val_loss 0.6785
Mechanistic early stopping at epoch 10


In [6]:
#Freeze weights in Mechanistic MLP Model and train CBCL model as correction model
for param in mech_model.parameters():
    param.requires_grad=False
mech_model.eval()

optimizer_cbcl = torch.optim.Adam(cbcl_model.parameters(), lr=1e-3)

best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

for epoch in range(100):
    cbcl_model.train()
    train_loss = 0.0
    n_train = 0

    for x_mech, x_cbcl, y in train_loader:
        x_mech = x_mech.to(device)
        x_cbcl = x_cbcl.to(device)
        y = y.to(device)

        optimizer_cbcl.zero_grad()

        with torch.no_grad():
            mech_logits = mech_model(x_mech)

        cbcl_logits = cbcl_model(x_cbcl)
        final_logits = mech_logits + cbcl_logits

        loss = criterion(final_logits, y)
        loss.backward()
        optimizer_cbcl.step()

        train_loss += loss.item() * y.size(0)
        n_train += y.size(0)

    train_loss /= n_train

    cbcl_model.eval()
    val_loss = 0.0
    n_val = 0

    with torch.no_grad():
        for x_mech, x_cbcl, y in val_loader:
            x_mech = x_mech.to(device)
            x_cbcl = x_cbcl.to(device)
            y = y.to(device)

            mech_logits = mech_model(x_mech)
            cbcl_logits = cbcl_model(x_cbcl)
            final_logits = mech_logits + cbcl_logits #Note: mech_logits can't change and 

            loss = criterion(final_logits, y)

            val_loss += loss.item() * y.size(0)
            n_val += y.size(0)

    val_loss /= n_val

    print(f"[CBCL correction] epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f}")

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = copy.deepcopy(cbcl_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"CBCL correction early stopping at epoch {epoch}")
            break

if best_state is not None:
    cbcl_model.load_state_dict(best_state)


[CBCL correction] epoch 000 | train_loss 0.2887 | val_loss 0.2909
[CBCL correction] epoch 001 | train_loss 0.2810 | val_loss 0.2886
[CBCL correction] epoch 002 | train_loss 0.2791 | val_loss 0.2883
[CBCL correction] epoch 003 | train_loss 0.2787 | val_loss 0.2882
[CBCL correction] epoch 004 | train_loss 0.2784 | val_loss 0.2881
[CBCL correction] epoch 005 | train_loss 0.2782 | val_loss 0.2879
[CBCL correction] epoch 006 | train_loss 0.2781 | val_loss 0.2880
[CBCL correction] epoch 007 | train_loss 0.2779 | val_loss 0.2879
[CBCL correction] epoch 008 | train_loss 0.2779 | val_loss 0.2881
[CBCL correction] epoch 009 | train_loss 0.2776 | val_loss 0.2887
[CBCL correction] epoch 010 | train_loss 0.2778 | val_loss 0.2883
[CBCL correction] epoch 011 | train_loss 0.2776 | val_loss 0.2883
[CBCL correction] epoch 012 | train_loss 0.2775 | val_loss 0.2883
[CBCL correction] epoch 013 | train_loss 0.2775 | val_loss 0.2882
[CBCL correction] epoch 014 | train_loss 0.2774 | val_loss 0.2881
[CBCL corr

In [7]:
#Obtain metrics on test dataset
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

mech_model.eval()
cbcl_model.eval()

mech_probs = []
final_probs = []
test_labels = []

with torch.no_grad():
    for x_mech, x_cbcl, y in test_loader:
        x_mech = x_mech.to(device)
        x_cbcl = x_cbcl.to(device)
        y = y.to(device)

        mech_logits = mech_model(x_mech)
        cbcl_logits = cbcl_model(x_cbcl)
        final_logits = mech_logits + cbcl_logits

        mech_prob = F.softmax(mech_logits, dim=1)[:, 1]
        final_prob = F.softmax(final_logits, dim=1)[:, 1]

        mech_probs.extend(mech_prob.cpu().numpy())
        final_probs.extend(final_prob.cpu().numpy())
        test_labels.extend(y.cpu().numpy())

mech_auc = roc_auc_score(test_labels, mech_probs)
mech_auprc = average_precision_score(test_labels, mech_probs)

final_auc = roc_auc_score(test_labels, final_probs)
final_auprc = average_precision_score(test_labels, final_probs)

#Use these values for bootstrapping
y_true_ap3 = np.array(test_labels)
y_prob_ap3 = np.array(final_probs)


print("Mechanistic-only test AUC:", mech_auc)
print("Mechanistic-only test AUPRC:", mech_auprc)
print("Residual model test AUC:", final_auc)
print("Residual model test AUPRC:", final_auprc)

Mechanistic-only test AUC: 0.6749175715656163
Mechanistic-only test AUPRC: 0.20175955626812547
Residual model test AUC: 0.7783420800180577
Residual model test AUPRC: 0.3002767393044353


# Computing Integrated Gradients

In [8]:
import torch
import torch.nn as nn

feature_names = ABCD_train.columns.tolist()

# Full-matrix feature indices
cbcl_indxs = [ABCD_train.columns.get_loc(col) for col in cbcl_cols]
mech_indxs = [i for i in range(len(feature_names)) if i not in cbcl_indxs]

cbcl_features = [feature_names[i] for i in cbcl_indxs]
mech_features = [feature_names[i] for i in mech_indxs]

#Combine the two separate models to evaluate integrated gradients
class ResidualWrapper(nn.Module):
    def __init__(self, mech_model, cbcl_model, mech_indxs, cbcl_indxs):
        super().__init__()
        self.mech_model = mech_model
        self.cbcl_model = cbcl_model
        self.mech_indxs = mech_indxs
        self.cbcl_indxs = cbcl_indxs

    def forward(self, x):
        x_mech = x[:, self.mech_indxs]
        x_cbcl = x[:, self.cbcl_indxs]

        mech_logits = self.mech_model(x_mech)
        cbcl_logits = self.cbcl_model(x_cbcl)

        final_logits = mech_logits + cbcl_logits

        # Return positive-class log-odds
        log_odds = final_logits[:, 1] - final_logits[:, 0]

        return log_odds

residual_model = ResidualWrapper(
    mech_model=mech_model,
    cbcl_model=cbcl_model,
    mech_indxs=mech_indxs,
    cbcl_indxs=cbcl_indxs
).to(device)

residual_model.eval()

ResidualWrapper(
  (mech_model): MLP(
    (net): Sequential(
      (0): Linear(in_features=149, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=2, bias=True)
    )
  )
  (cbcl_model): MLP(
    (net): Sequential(
      (0): Linear(in_features=8, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=2, bias=True)
    )
  )
)

In [9]:
from captum.attr import IntegratedGradients
import numpy as np
import torch

feature_names = ABCD_train.columns.tolist()

X_test_full = ABCD_test.drop(columns=["is_quartile_4"])[feature_names].copy()

X_test_tensor = torch.tensor(
    X_test_full.iloc[:200].to_numpy(dtype=np.float32),
    dtype=torch.float32
).to(device)

baseline = torch.zeros_like(X_test_tensor)

ig = IntegratedGradients(residual_model)

attributions, delta = ig.attribute(
    inputs=X_test_tensor,
    baselines=baseline,
    n_steps=50,
    return_convergence_delta=True
)

ig_values = attributions.detach().cpu().numpy()

mech_ig = ig_values[:, mech_indxs]
cbcl_ig = ig_values[:, cbcl_indxs]

mech_net = np.mean(np.abs(np.sum(mech_ig, axis=1)))
cbcl_net = np.mean(np.abs(np.sum(cbcl_ig, axis=1)))

ig_ratio_net = mech_net / cbcl_net if cbcl_net != 0 else np.inf

print("Full Test Dataset (with CBCL)")
print("Mechanistic Net IG:", mech_net)
print("CBCL Net IG:", cbcl_net)
print("Mechanistic / CBCL Net IG Ratio:", ig_ratio_net)


#Calculate Integrated Gradients for when CBCL features are removed from testing dataset (aka set to 0)
X_test_zero_cbcl = X_test_full.copy()

# Set all CBCL columns to 0
X_test_zero_cbcl.iloc[:, cbcl_indxs] = 0

X_test_zero_cbcl_tensor = torch.tensor(
    X_test_zero_cbcl.iloc[:200].to_numpy(dtype=np.float32),
    dtype=torch.float32
).to(device)

# Since the input has CBCL = 0, use a zero baseline too
baseline_zero_cbcl = torch.zeros_like(X_test_zero_cbcl_tensor)

attributions_zero_cbcl, delta_zero_cbcl = ig.attribute(
    inputs=X_test_zero_cbcl_tensor,
    baselines=baseline_zero_cbcl,
    n_steps=50,
    return_convergence_delta=True
)

ig_values_zero_cbcl = attributions_zero_cbcl.detach().cpu().numpy()

mech_ig_zero_cbcl = ig_values_zero_cbcl[:, mech_indxs]
cbcl_ig_zero_cbcl = ig_values_zero_cbcl[:, cbcl_indxs]

mech_net_zero_cbcl = np.mean(np.abs(np.sum(mech_ig_zero_cbcl, axis=1)))
cbcl_net_zero_cbcl = np.mean(np.abs(np.sum(cbcl_ig_zero_cbcl, axis=1)))

ig_ratio_net_zero_cbcl = (
    mech_net_zero_cbcl / cbcl_net_zero_cbcl
    if cbcl_net_zero_cbcl != 0
    else np.inf
)

print("\n\nTest Dataset without CBCL Features")
print("Mechanistic Net IG:", mech_net_zero_cbcl)
print("CBCL Net IG:", cbcl_net_zero_cbcl)
print("Mechanistic / CBCL Net IG Ratio:", ig_ratio_net_zero_cbcl)

Full Test Dataset (with CBCL)
Mechanistic Net IG: 1.7596383
CBCL Net IG: 1.2801013
Mechanistic / CBCL Net IG Ratio: 1.3746086


Test Dataset without CBCL Features
Mechanistic Net IG: 1.7596383
CBCL Net IG: 0.0
Mechanistic / CBCL Net IG Ratio: inf


In [10]:
import pandas as pd
import numpy as np

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_ig": np.mean(np.abs(ig_values), axis=0),
    "mean_signed_ig": np.mean(ig_values, axis=0)
})

feature_importance = feature_importance.sort_values(
    "mean_abs_ig",
    ascending=False
)

print(feature_importance.head(20))

feature_importance_0 = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_ig": np.mean(np.abs(ig_values_zero_cbcl), axis=0),
    "mean_signed_ig": np.mean(ig_values_zero_cbcl, axis=0)
})

feature_importance_0 = feature_importance_0.sort_values(
    "mean_abs_ig",
    ascending=False
)

print(feature_importance_0.head(20))
feature_importance.head(20)

                       feature  mean_abs_ig  mean_signed_ig
5     cbcl_scr_syn_attention_t     0.366162       -0.288019
3        cbcl_scr_syn_social_t     0.288650       -0.251947
7    cbcl_scr_syn_aggressive_t     0.280725       -0.257023
0        cbcl_scr_syn_anxdep_t     0.225598       -0.132294
6     cbcl_scr_syn_rulebreak_t     0.200425       -0.156594
15                  total_core     0.190280       -0.058336
1       cbcl_scr_syn_withdep_t     0.152088       -0.057629
4       cbcl_scr_syn_thought_t     0.136596       -0.082842
43              prosocial_q1_p     0.125286       -0.042508
102           sleepdisturb22_p     0.105146       -0.050416
84             sleepdisturb4_p     0.093240       -0.037522
16               fam_enviro1_p     0.085119       -0.022414
56                  school_9_y     0.078938       -0.004593
103           sleepdisturb23_p     0.073941       -0.041858
20               fam_enviro5_p     0.063208       -0.021744
57                 school_10_y     0.063

,feature,mean_abs_ig,mean_signed_ig
5,cbcl_scr_syn_attention_t,0.366162,-0.288019
3,cbcl_scr_syn_social_t,0.288650,-0.251947
7,cbcl_scr_syn_aggressive_t,0.280725,-0.257023
0,cbcl_scr_syn_anxdep_t,0.225598,-0.132294
6,cbcl_scr_syn_rulebreak_t,0.200425,-0.156594
15,total_core,0.190280,-0.058336
1,cbcl_scr_syn_withdep_t,0.152088,-0.057629
4,cbcl_scr_syn_thought_t,0.136596,-0.082842
43,prosocial_q1_p,0.125286,-0.042508
102,sleepdisturb22_p,0.105146,-0.050416


In [14]:
#Save Models with dataset settings and save integrated gradient tables
checkpoint = {
    "mech_model_state_dict": mech_model.state_dict(),
    "cbcl_model_state_dict": cbcl_model.state_dict(),
    "feature_names": feature_names,
    "mech_indxs": mech_indxs,
    "cbcl_indxs": cbcl_indxs,
    "mech_cols": mech_cols,
    "cbcl_cols": cbcl_cols,
    "mech_hidden_dims": (128, 64),
    "cbcl_hidden_dims": (8,),
    "num_classes": 2
}

torch.save(checkpoint, "Insert Here")

feature_importance["Label"] = "Full Dataset Model"
feature_importance_0["Label"] = "Test Dataset without CBCL"

combined_feature_importance = pd.concat([feature_importance, feature_importance_0], axis=0, ignore_index=True)
combined_feature_importance.to_csv("Insert Here", index=False)

In [ ]:
#Reload saved model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load("Insert Here", map_location=device)

feature_names = checkpoint["feature_names"]
mech_indxs = checkpoint["mech_indxs"]
cbcl_indxs = checkpoint["cbcl_indxs"]
mech_cols = checkpoint["mech_cols"]
cbcl_cols = checkpoint["cbcl_cols"]

mech_model = MLP(
    input_dim=len(mech_indxs),
    hidden_dims=checkpoint["mech_hidden_dims"],
    num_classes=checkpoint["num_classes"]
).to(device)

cbcl_model = MLP(
    input_dim=len(cbcl_indxs),
    hidden_dims=checkpoint["cbcl_hidden_dims"],
    num_classes=checkpoint["num_classes"]
).to(device


#Load weights
mech_model.load_state_dict(checkpoint["mech_model_state_dict"])
cbcl_model.load_state_dict(checkpoint["cbcl_model_state_dict"])

mech_model.eval()
cbcl_model.eval()
